<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">📊</div><div style="color:white;font-size:1.5rem;font-weight:700;">Topic Explorer</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Discover hidden themes using LDA topic modelling<br><a href="https://ladal.edu.au/topicmodels.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to the <b>MyTexts</b> folder (Step 2).</li><li>Set the <b>number of topics</b> and other settings in the configuration cell (Step 3).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download results from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> After uploading your files and editing the configuration cell below, run all cells at once by clicking <b>Kernel &#x2192; Restart &amp; Run All</b> in the menu above. You only need to run one step manually: uploading files via the file browser panel on the left.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Setup — loads helper functions used throughout the notebook.
# This cell runs automatically when you use Kernel > Restart & Run All.

suppressPackageStartupMessages(library(IRdisplay))

# Colour-coded feedback helpers
.ok   <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#eafaf1;border-left:4px solid #27ae60;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2705; ', msg, '</div>'))
.warn <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff8e1;border-left:4px solid #f0a500;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x26A0;&#xFE0F; ', msg, '</div>'))
.err  <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff0f0;border-left:4px solid #e74c3c;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x274C; ', msg, '</div>'))
.info <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#f4f0f8;border-left:4px solid #51247a;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2139;&#xFE0F; ', msg, '</div>'))
.prog <- function(label, pct) IRdisplay::display_html(paste0(
  '<div style="font-family:sans-serif;font-size:.85rem;margin:.3rem 0;">',
  '<b style="color:#51247a;">', label, '</b>',
  '<div style="background:#e8e4f0;border-radius:4px;height:7px;margin-top:3px;">',
  '<div style="background:#51247a;width:', round(pct), '%;height:7px;',
  'border-radius:4px;"></div></div></div>'))

# Load plain-text files from a folder
load_texts <- function(folder = "notebooks/MyTexts") {
  # Create folder if it does not exist (git does not track empty dirs)
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  files <- list.files(folder, pattern = "(?i)\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop(
    "No .txt files found in '", folder, "'\n\n",
    "Please make sure:\n",
    "  1. You opened the correct folder in the file browser (left panel)\n",
    "  2. You dragged .txt files into that folder\n",
    "  3. Files have a .txt extension (not .docx, .pdf, etc.)\n",
    "  4. You ran Kernel > Restart & Run All AFTER uploading")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# Save named character vector as .txt files
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# Save data frame as Excel
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

.ok("Setup complete.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts to the MyTexts folder</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left of the screen (click the folder icon if it is not visible).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the notebook.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────
# Edit the values below, then run this cell (Shift+Enter).

n_topics        <- 5     # number of topics to find (try 3-10 for small corpora)
n_terms         <- 10    # number of top terms to show per topic
n_iterations    <- 1000  # LDA iterations (higher = more accurate but slower)
remove_stopwords <- TRUE # remove common English words before modelling
min_doc_freq    <- 2     # minimum document frequency for a word to be included


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Topic modelling — runs automatically.
suppressPackageStartupMessages(library(topicmodels))
suppressPackageStartupMessages(library(quanteda))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(writexl))

texts <- tryCatch(load_texts("notebooks/MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })
if (!is.null(texts)) {
  .info(paste0("Loaded ", length(texts), " file(s)."))
  .prog("Building document-term matrix...", 15)
  toks <- tokens(corpus(texts),
                 remove_punct = TRUE, remove_numbers = TRUE,
                 remove_symbols = TRUE) |> tokens_tolower()
  if (remove_stopwords)
    toks <- tokens_remove(toks, stopwords("en"), padding = FALSE)
  dfm_obj <- dfm(toks) |> dfm_trim(min_docfreq = min_doc_freq)
  dfm_obj  <- dfm_obj[rowSums(dfm_obj) > 0, ]
  .ok(paste0("DTM: <b>", ndoc(dfm_obj), " documents</b> | <b>",
             nfeat(dfm_obj), " features</b>"))
  if (ndoc(dfm_obj) < n_topics)
    stop("More topics (", n_topics, ") than documents (",
         ndoc(dfm_obj), "). Please reduce n_topics.")
  .prog(paste0("Fitting LDA (", n_topics, " topics, ",
               n_iterations, " iterations)..."), 40)
  dtm <- quanteda::convert(dfm_obj, to = "topicmodels")
  lda <- topicmodels::LDA(dtm, k = n_topics, method = "Gibbs",
           control = list(seed = 42, iter = n_iterations, burnin = 100))
  .prog("Extracting top terms...", 78)
  terms_df <- as.data.frame(topicmodels::terms(lda, n_terms))
  colnames(terms_df) <- paste0("Topic_", seq_len(n_topics))
  terms_df$Rank <- seq_len(n_terms)
  terms_df <- terms_df[, c("Rank", paste0("Topic_", seq_len(n_topics)))]
  print(terms_df)
  .prog("Plotting topic-document distribution...", 88)
  gamma_df <- as.data.frame(topicmodels::posterior(lda)$topics)
  colnames(gamma_df) <- paste0("Topic_", seq_len(n_topics))
  gamma_df$document  <- rownames(gamma_df)
  p <- gamma_df |>
    tidyr::pivot_longer(-document, names_to = "topic", values_to = "prob") |>
    ggplot(aes(x = topic, y = prob, fill = topic)) +
    geom_col(show.legend = FALSE, width = .7) +
    facet_wrap(~document) +
    scale_fill_viridis_d(option = "plasma") +
    coord_flip() + theme_bw(base_size = 11) +
    labs(title = "Topic distribution across documents",
         x = NULL, y = "Probability")
  print(p)
  save_excel(terms_df, "topic_terms.xlsx")
  save_excel(gamma_df, "topic_documents.xlsx")
  dir.create("notebooks/MyOutput", showWarnings = FALSE, recursive = TRUE)
  ggsave("notebooks/MyOutput/topic_plot.png",
         bg = "white", width = 9, height = 5, dpi = 200)
  .ok("Saved topic_terms.xlsx, topic_documents.xlsx, topic_plot.png.")
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


---

### Citation

Schweinberger, Martin. (2025). *LADAL Topic Explorer*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025topics,
  author = {Schweinberger, Martin},
  title  = {LADAL Topic Explorer},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()